In [ ]:
## link to investments. prioritse source that has investment & capacity. 

In [40]:
import pandas as pd
import sys; sys.path.append("..")
from src.vote_helpers import fill_single_product_lv2, parse_capacity_value
from mongo_client import mongo_client, capacities_collection
from collections import defaultdict

try:
    mongo_client.admin.command("ping")
    print("✅ Connected to MongoDB Atlas!")
except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}")
    raise

df = pd.read_excel("storage/output/filtered_projects_jul16.xlsx")
df['date'] = pd.to_datetime(df['date'], format='%Y-%m')

# fill product_lv2 across multiple NA rows
df = df.groupby("cluster_id").apply(fill_single_product_lv2).reset_index(drop=True)

# normalise capacity values
df["capacity_normalized"] = df["capacity_normalized"].apply(parse_capacity_value)

print(len(df))

print(len(df[(df["phase"] == "unclear")]))

print(len(df[df["status"] == "unclear"]))

print(len(df[df["date"].isna()]))

# drop all cases where phase | status are unclear

df = df[(df["phase"] != "unclear") & (df["status"] != "unclear")].copy()

df = df[~df["date"].isna()].copy()

print(len(df))

✅ Connected to MongoDB Atlas!
345
26
1
47
277


/var/folders/2d/7gfcftkd69s2z741sq98f0sc0000gn/T/ipykernel_35705/3256782809.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("cluster_id").apply(fill_single_product_lv2).reset_index(drop=True)


In [43]:
df["clutser_id"].unique()

KeyError: 'clutser_id'

In [44]:
df[df["cluster_id"] == 43]

,inst_canon,city_key,adm1,adm2,iso2,cluster_id,product_lv1,product_lv2,product,capacity_normalized,capacity_metric_normalized,status,phase,date,article_id
23,man,nuremberg,Bavaria,Middle Franconia,DE,43,battery,module_pack,high-voltage batteries,2.5,GWh,announced,greenfield,2024-11-01,67f92305f431fddd61d56051
24,man,nuremberg,Bavaria,Middle Franconia,DE,43,battery,module_pack,high-voltage batteries,5.0,GWh,announced,expansion,2024-11-01,67f92305f431fddd61d56051


In [ ]:
import logging

# Remove all handlers associated with the root logger object (to avoid duplicate logs in Jupyter)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Set up logging to file and console
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("votes_log.log", mode='w'),  # opening in write mode (overwriting each time)
        logging.StreamHandler()  # This keeps output in the notebook as well
    ]
)

# Example usage
logging.info("INITIATING LOGGING")
unique_clusters = df["cluster_id"].unique().tolist()
logging.info(f"Votes for {len(unique_clusters)} clusters.")
status_counts = defaultdict(int)

for cluster in unique_clusters:

    logging.info(f"Assessing votes for cluster_id: {cluster}")

    df_cluster = df[df["cluster_id"] == cluster].copy()
    unique_statuses = df_cluster["status"].unique()
    unique_phases = df_cluster["phase"].unique()

    owner = df_cluster["inst_canon"].unique()[0]
    iso2 = df_cluster["iso2"].unique()[0]
    adm1 = df_cluster["adm1"].unique()[0]
    product_lv1 = df_cluster["product_lv1"].unique()[0]

    if "announced" in unique_statuses and len(unique_statuses) <2:
        logging.info("⚪️ ANNOUNCED case")
        status_counts["announced"] += 1

        # only announcement so we ignore expansions 
        df_canon = df_cluster[df_cluster["phase"] != "expansion"]
        if len(df_canon) < 1:
            logging.info("🔴 ERROR - no greenfield phase noted at facility, only expansions. DROPPED.")
            status_counts["announced-dropped-only-expansions"] += 1
            continue

        dt_announce = df_canon["date"].min()
        phase = "greenfield"

        # find row with the latest date & take capacity
        row_latest = df_canon.loc[df_canon["date"].idxmax()]
        capacity_latest = row_latest["capacity_normalized"]
        
        dt_construct = None
        dt_operation = None

    elif "under construction" in unique_statuses and "operational" not in unique_statuses:
        logging.info("🔵 UNDER CONSTRUCTION case")
        status_counts["under-construction"] += 1

        # only under construction so we ignore expansions 
        df_canon = df_cluster[df_cluster["phase"] != "expansion"]
        if len(df_canon) < 1:
            logging.info("🔴 ERROR - no greenfield phase noted at facility, only expansions. DROPPED.")
            status_counts["under-construction-dropped-only-expansions"] += 1
            continue

        dt_announce = df_canon["date"].min()
        status = "under construction"
        phase = "greenfield"

        # Find row with the latest date & take capacity (for under construction cases)
        df_construct = df_canon[df_canon["status"] == "under construction"].copy()
        dt_construct = df_construct["date"].min()
        row_latest_construction = df_construct.loc[df_construct["date"].idxmax()]
        capacity_latest = row_latest["capacity_normalized"]
        
        dt_operation = None

    elif "operational" in unique_statuses:
        logging.info("🟢 OPERATIONAL case")
        status_counts["operational"] += 1

        # cases where there are no expansions
        if "expansion" not in unique_phases:
            logging.info("No expansions sub-case")
            status_counts["operational-no-expansions"] += 1

            # earliest date taken for dt_announce
            dt_announce = df_cluster["date"].min()    #the earliest date mentioned
            status = "operational"
            phase = "greenfield"

            # earliest date where status == under construction taken for dt_construct
            df_construct = df_cluster[df_cluster["status"] == "under construction"].copy()
            dt_construct = df_construct["date"].min()

            # earliest date where status == operational taken for dt_operation
            df_operation = df_cluster[df_cluster["status"] == "operational"].copy()
            dt_operation = df_cluster["date"].min()

            # take operational capacity from most recent article
            row_latest_operation = df_operation.loc[df_operation["date"].idxmax()]
            capacity_latest = row_latest["capacity_normalized"]

        if "expansion" in unique_phases:
            logging.info("With expansions sub-case")
            status_counts["operational-with-expansions"] += 1


    else:
        logging.info("Group not assigned")
        status_counts["unassigned"] += 1

# # final dictionary to insert to Mongo
#     mongo_entry = {
#         "owner": owner,
#         "iso2": iso2,
#         "adm1": adm1,
#         "product_lv1": product_lv1,
#         #"product": product_dict,
#         "dt_announce": dt_announce.strftime("%Y-%m") if pd.notnull(dt_announce) else None,
#         "status": "announced",
#         "phase": phase,
#         "capacity": capacity_latest,
#         "dt_construct": dt_construct.strftime("%Y-%m") if pd.notnull(dt_construct) else None,
#         "dt_operation": dt_operation.strftime("%Y-%m") if pd.notnull(dt_operation) else None
#     }

#     capacities_collection.insert_one(mongo_entry)
#     logging.info("✅ Inserted announced project into MongoDB.")

logging.info("📊 Final Status Counts:")
for status, count in status_counts.items():
    logging.info(f"{status}: {count}")

2025-07-17 16:55:59,891 - INFO - INITIATING LOGGING
2025-07-17 16:55:59,899 - INFO - Votes for 71 clusters.
2025-07-17 16:55:59,901 - INFO - Assessing votes for cluster_id: 2
2025-07-17 16:55:59,909 - INFO - 🔵 UNDER CONSTRUCTION case
2025-07-17 16:55:59,914 - INFO - Assessing votes for cluster_id: 35
2025-07-17 16:55:59,914 - INFO - 🔵 UNDER CONSTRUCTION case
2025-07-17 16:55:59,915 - INFO - Assessing votes for cluster_id: 37
2025-07-17 16:55:59,916 - INFO - 🔵 UNDER CONSTRUCTION case
2025-07-17 16:55:59,916 - INFO - Assessing votes for cluster_id: 39
2025-07-17 16:55:59,917 - INFO - 🔵 UNDER CONSTRUCTION case
2025-07-17 16:55:59,918 - INFO - Assessing votes for cluster_id: 41
2025-07-17 16:55:59,918 - INFO - 🟢 OPERATIONAL case
2025-07-17 16:55:59,919 - INFO - No expansions sub-case
2025-07-17 16:55:59,920 - INFO - Assessing votes for cluster_id: 42
2025-07-17 16:55:59,920 - INFO - 🟢 OPERATIONAL case
2025-07-17 16:55:59,921 - INFO - With expansions sub-case
2025-07-17 16:55:59,921 - INFO 

In [32]:
df[df["cluster_id"] == 41]

,inst_canon,city_key,adm1,adm2,iso2,cluster_id,product_lv1,product_lv2,product,capacity_normalized,capacity_metric_normalized,status,phase,date,article_id
21,fenecon,iggensbach,Bavaria,Lower Bavaria,DE,41,battery,NaN,large-scale stationary storage systems,0.025,GWh,operational,greenfield,2024-04-01,67f922e7f431fddd61d55fee
